# Проверка пересечений ИНН: CRM vs Problem Status

| Источник | Файл на сервере | Ключевое поле ИНН | Поле даты |
|----------|----------------|--------------------|-----------|
| CRM | `CRM_segment_with_segment.csv` | `X_INN` | `IDENTIFICATION_DATE` (ГГГГ-ММ-ДД) |
| Problem Status | `inn_problem_info.xlsx` | `X_INN` | `SOL_DATE_UOB` (ДД.ММ.ГГГГ) |

**Фильтр Problem Status:** `VAL_1` = *Признание задолжен. проблемной*  
**Период:** 2023–2025 включительно

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "../sources"  # <-- путь к папке с данными

CRM_FILE = f"{DATA_DIR}/CRM_segment_with_segment.csv"
PS_FILE  = f"{DATA_DIR}/inn_problem_info.xlsx"

PS_DECISION  = "Признание задолжен. проблемной"
DATE_FROM    = pd.Timestamp("2023-01-01")
DATE_TO      = pd.Timestamp("2025-12-31")


def normalize_inn(s):
    """Нормализует ИНН: убирает .0, ведущие нули, дополняет до 10 или 12 знаков.

    Excel хранит ИНН как число, поэтому при чтении:
    - ИНН может быть float: 101012203.0 → '101012203' → '0101012203'
    - Могут пропасть ведущие нули: '101012203' вместо '0101012203'
    Функция приводит все варианты к единому формату.
    """
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


print("Параметры заданы.")

---
## Этап 1. Загрузка данных

Каждая выгрузка загружается в отдельной ячейке, чтобы при изменениях не перечитывать обе.

In [ ]:
# --- CRM ---
df_crm = pd.read_csv(CRM_FILE, sep=";", encoding="utf-8-sig", dtype=str, low_memory=False)
df_crm["inn_raw"] = df_crm["X_INN"].astype(str).str.strip()
df_crm["inn"]     = df_crm["X_INN"].apply(normalize_inn)

print(f"CRM загружен: {len(df_crm):,} строк")
print(f"Уникальных ИНН (raw): {df_crm['inn_raw'].nunique():,}")
print(f"Уникальных ИНН (norm): {df_crm['inn'].nunique():,}")
print(f"Колонки: {list(df_crm.columns[:5])} ... ({len(df_crm.columns)} всего)")

In [ ]:
# --- Problem Status (Excel) ---
df_ps = pd.read_excel(PS_FILE, dtype=str)
df_ps["inn_raw"] = df_ps["X_INN"].astype(str).str.strip()
df_ps["inn"]     = df_ps["X_INN"].apply(normalize_inn)

print(f"Problem Status загружен: {len(df_ps):,} строк")
print(f"Уникальных ИНН (raw): {df_ps['inn_raw'].nunique():,}")
print(f"Уникальных ИНН (norm): {df_ps['inn'].nunique():,}")
print(f"Колонки: {list(df_ps.columns[:5])} ... ({len(df_ps.columns)} всего)")
print(f"\nЗначения VAL_1 (топ-10):")
print(df_ps["VAL_1"].value_counts().head(10).to_string())

---
## Этап 2. Фильтрация Problem Status по типу решения

Оставляем только записи с `VAL_1` = *Признание задолжен. проблемной*.

In [ ]:
df_ps["VAL_1"] = df_ps["VAL_1"].astype(str).str.strip()
df_ps_filtered = df_ps[df_ps["VAL_1"] == PS_DECISION].copy()

total_rows   = len(df_ps_filtered)
unique_inns  = df_ps_filtered["inn"].nunique()

print("=" * 60)
print(f"Фильтр: VAL_1 = '{PS_DECISION}'")
print("=" * 60)
print(f"  Строк после фильтра:   {total_rows:,}")
print(f"  Уникальных ИНН:        {unique_inns:,}")
print(f"  (было до фильтра:      {len(df_ps):,} строк, {df_ps['inn'].nunique():,} ИНН)")

---
## Проверка: влияние ведущих нулей на пересечение ИНН

Excel может хранить ИНН как число, из-за чего ведущие нули пропадают.  
Сравниваем пересечение **без** нормализации (raw) и **с** нормализацией, чтобы оценить масштаб потерь.

In [ ]:
inn_crm_raw = set(df_crm["inn_raw"].dropna().unique())
inn_ps_raw  = set(df_ps_filtered["inn_raw"].dropna().unique())

inn_crm_norm = set(df_crm["inn"].dropna().unique())
inn_ps_norm  = set(df_ps_filtered["inn"].dropna().unique())

overlap_raw  = inn_ps_raw & inn_crm_raw
overlap_norm = inn_ps_norm & inn_crm_norm

gained = len(overlap_norm) - len(overlap_raw)

print("=" * 65)
print("ВЛИЯНИЕ НОРМАЛИЗАЦИИ ИНН НА ПЕРЕСЕЧЕНИЕ")
print("=" * 65)
print(f"  {'':40s} {'Raw':>10s} {'Norm':>10s}")
print(f"  {'\u2014' * 60}")
print(f"  {'Уникальных ИНН в CRM':<40s} {len(inn_crm_raw):>10,} {len(inn_crm_norm):>10,}")
print(f"  {'Уникальных ИНН в PS':<40s} {len(inn_ps_raw):>10,} {len(inn_ps_norm):>10,}")
print(f"  {'Пересечение (PS \u2229 CRM)':<40s} {len(overlap_raw):>10,} {len(overlap_norm):>10,}")
print(f"  {'В PS, но НЕТ в CRM':<40s} {len(inn_ps_raw - inn_crm_raw):>10,} {len(inn_ps_norm - inn_crm_norm):>10,}")
print(f"\n  \u0414ополнительно найдено благодаря нормализации: {gained:,}")

if gained > 0:
    recovered_by_norm = (inn_ps_norm & inn_crm_norm) - overlap_raw
    print(f"\n  Примеры ИНН, найденных только после нормализации (до 10):")
    for inn in sorted(recovered_by_norm)[:10]:
        raw_crm = df_crm.loc[df_crm["inn"] == inn, "inn_raw"].iloc[0] if len(df_crm[df_crm["inn"] == inn]) else "\u2014"
        raw_ps  = df_ps_filtered.loc[df_ps_filtered["inn"] == inn, "inn_raw"].iloc[0] if len(df_ps_filtered[df_ps_filtered["inn"] == inn]) else "\u2014"
        print(f"    norm: {inn}  |  CRM raw: {raw_crm}  |  PS raw: {raw_ps}")
elif gained == 0:
    print("\n  \u2705 Нормализация не повлияла \u2014 ведущие нули не вызывают расхождений.")

---
## Этап 3. Сравнение ИНН БЕЗ фильтра по дате

Сравниваем полные наборы ИНН: все ИНН из отфильтрованного Problem Status vs все ИНН из CRM.

In [ ]:
inn_crm_full = set(df_crm["inn"].dropna().unique())
inn_ps_full  = set(df_ps_filtered["inn"].dropna().unique())

overlap_full    = inn_ps_full & inn_crm_full
ps_only_full    = inn_ps_full - inn_crm_full
crm_only_full   = inn_crm_full - inn_ps_full

print("=" * 60)
print("СРАВНЕНИЕ БЕЗ ФИЛЬТРА ПО ДАТЕ (все данные)")
print("=" * 60)
print(f"  ИНН в CRM:                        {len(inn_crm_full):>8,}")
print(f"  ИНН в Problem Status:              {len(inn_ps_full):>8,}")
print(f"  \u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014\u2014")
print(f"  Пересечение (PS \u2229 CRM):             {len(overlap_full):>8,}")
print(f"  Только в PS (нет в CRM):           {len(ps_only_full):>8,}")
print(f"  Только в CRM (нет в PS):           {len(crm_only_full):>8,}")
print(f"\n  Доля PS, покрытая CRM:             {len(overlap_full)/len(inn_ps_full)*100:>7.1f}%")

if ps_only_full:
    print(f"\n  Примеры ИНН из PS, которых нет в CRM (до 5):")
    for inn in sorted(ps_only_full)[:5]:
        n = len(df_ps_filtered[df_ps_filtered["inn"] == inn])
        print(f"    {inn}  \u2192  {n} строк в PS")

---
## Этап 4. Фильтрация по периоду 2023–2025

- **CRM:** `IDENTIFICATION_DATE` (формат ГГГГ-ММ-ДД)  
- **Problem Status:** `SOL_DATE_UOB` (формат ДД.ММ.ГГГГ)  

Приводим обе даты к единому формату `datetime`.

In [ ]:
df_crm["dt"] = pd.to_datetime(df_crm["IDENTIFICATION_DATE"], errors="coerce")
df_crm_period = df_crm[(df_crm["dt"] >= DATE_FROM) & (df_crm["dt"] <= DATE_TO)].copy()

df_ps_filtered["dt"] = pd.to_datetime(df_ps_filtered["SOL_DATE_UOB"], dayfirst=True, errors="coerce")
df_ps_period = df_ps_filtered[
    (df_ps_filtered["dt"] >= DATE_FROM) & (df_ps_filtered["dt"] <= DATE_TO)
].copy()

print("=" * 60)
print(f"ФИЛЬТРАЦИЯ ПО ПЕРИОДУ {DATE_FROM:%d.%m.%Y} \u2013 {DATE_TO:%d.%m.%Y}")
print("=" * 60)
print(f"  CRM:")
print(f"    Было: {len(df_crm):>8,} строк, {df_crm['inn'].nunique():>6,} ИНН")
print(f"    Стало: {len(df_crm_period):>7,} строк, {df_crm_period['inn'].nunique():>6,} ИНН")
print(f"\n  Problem Status:")
print(f"    Было: {len(df_ps_filtered):>8,} строк, {df_ps_filtered['inn'].nunique():>6,} ИНН")
print(f"    Стало: {len(df_ps_period):>7,} строк, {df_ps_period['inn'].nunique():>6,} ИНН")

bad_crm = df_crm["dt"].isna().sum()
bad_ps  = df_ps_filtered["dt"].isna().sum()
if bad_crm or bad_ps:
    print(f"\n  \u26a0 Не удалось распарсить дату: CRM={bad_crm:,}, PS={bad_ps:,}")

---
## Этап 5. Финальное сравнение: с фильтром и без

Сводная статистика по вхождению ИНН из Problem Status в CRM.

In [ ]:
inn_crm_period = set(df_crm_period["inn"].dropna().unique())
inn_ps_period  = set(df_ps_period["inn"].dropna().unique())

overlap_period  = inn_ps_period & inn_crm_period
ps_only_period  = inn_ps_period - inn_crm_period

recovered = ps_only_period - ps_only_full

print("=" * 65)
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print("=" * 65)

header = f"{'Метрика':<45s} {'Без фильтра':>12s} {'2023\u20132025':>12s}"
print(header)
print("\u2014" * 65)
print(f"{'ИНН в Problem Status':<45s} {len(inn_ps_full):>12,} {len(inn_ps_period):>12,}")
print(f"{'ИНН в CRM':<45s} {len(inn_crm_full):>12,} {len(inn_crm_period):>12,}")
print(f"{'Пересечение (PS \u2229 CRM)':<45s} {len(overlap_full):>12,} {len(overlap_period):>12,}")
print(f"{'В PS, но НЕТ в CRM':<45s} {len(ps_only_full):>12,} {len(ps_only_period):>12,}")
print(f"{'Доля PS, покрытая CRM':<45s} {len(overlap_full)/len(inn_ps_full)*100:>11.1f}% {len(overlap_period)/len(inn_ps_period)*100:>11.1f}%")
print(f"\n  Выпали из-за фильтра по дате:  {len(recovered):,}")
print(f"  (есть в CRM без фильтра, но нет с фильтром 2023\u20132025)")

In [ ]:
if recovered:
    print("=" * 65)
    print(f"ИНН, выпавшие из CRM из-за фильтра по дате (до 10 примеров):")
    print("=" * 65)
    for inn in sorted(recovered)[:10]:
        crm_dates = pd.to_datetime(
            df_crm.loc[df_crm["inn"] == inn, "IDENTIFICATION_DATE"], errors="coerce"
        ).dropna()
        ps_dates = pd.to_datetime(
            df_ps_filtered.loc[df_ps_filtered["inn"] == inn, "SOL_DATE_UOB"],
            dayfirst=True, errors="coerce"
        ).dropna()
        crm_range = f"{crm_dates.min():%d.%m.%Y} \u2013 {crm_dates.max():%d.%m.%Y}" if len(crm_dates) else "\u2014"
        ps_range  = f"{ps_dates.min():%d.%m.%Y} \u2013 {ps_dates.max():%d.%m.%Y}" if len(ps_dates) else "\u2014"
        print(f"  ИНН {inn}")
        print(f"      CRM даты:  {crm_range}")
        print(f"      PS даты:   {ps_range}")

if ps_only_full:
    print(f"\n{'=' * 65}")
    print(f"ИНН, которых ДЕЙСТВИТЕЛЬНО нет в CRM (до 10 примеров):")
    print("=" * 65)
    for inn in sorted(ps_only_full)[:10]:
        ps_rows = df_ps_filtered[df_ps_filtered["inn"] == inn]
        n_rows = len(ps_rows)
        dates = pd.to_datetime(ps_rows["SOL_DATE_UOB"], dayfirst=True, errors="coerce").dropna()
        dt_range = f"{dates.min():%d.%m.%Y} \u2013 {dates.max():%d.%m.%Y}" if len(dates) else "\u2014"
        print(f"  ИНН {inn}  \u2192  {n_rows} строк, даты: {dt_range}")
else:
    print(f"\n  Все ИНН из Problem Status присутствуют в CRM.")

In [ ]:
summary = pd.DataFrame([
    {"Метрика": "ИНН в Problem Status", "Без фильтра": len(inn_ps_full), "2023\u20132025": len(inn_ps_period)},
    {"Метрика": "ИНН в CRM", "Без фильтра": len(inn_crm_full), "2023\u20132025": len(inn_crm_period)},
    {"Метрика": "Пересечение (PS \u2229 CRM)", "Без фильтра": len(overlap_full), "2023\u20132025": len(overlap_period)},
    {"Метрика": "В PS, но НЕТ в CRM", "Без фильтра": len(ps_only_full), "2023\u20132025": len(ps_only_period)},
    {"Метрика": "Доля PS, покрытая CRM",
     "Без фильтра": f"{len(overlap_full)/len(inn_ps_full)*100:.1f}%",
     "2023\u20132025": f"{len(overlap_period)/len(inn_ps_period)*100:.1f}%"},
])
display(summary.style.hide(axis="index"))